# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [2]:
#Libraries
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, classification_report, recall_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier

from xgboost import XGBClassifier

In [3]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [4]:
#Check Null values
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [5]:
# Drop the Null values
spaceship = spaceship.dropna()
spaceship.isnull().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

In [6]:
# Extract deck (first letter of Cabin)
spaceship["Cabin"] = spaceship["Cabin"].str[0]
spaceship.head(3)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False


In [7]:
spaceship_clean = spaceship.drop(columns=["PassengerId", "Name"], errors="ignore")
spaceship_clean.head(3)

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False


In [8]:
# Check the Transported (1) vs Not Transported (0)
((spaceship_clean['Transported'].value_counts() / len(spaceship_clean)).round(4) * 100)

Transported
True     50.36
False    49.64
Name: count, dtype: float64

In [9]:
# For non-numerical columns, do dummies.
spaceship_clean = pd.get_dummies(spaceship_clean, drop_first=True)
spaceship_clean.head(3)

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,False,False,False,False,False,False,True,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [10]:
features = spaceship_clean.drop(columns = ["Transported"])
target = spaceship_clean["Transported"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=0)

In [12]:
scaler = StandardScaler()

In [13]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
X_train_scaled

array([[ 0.22051462, -0.34704629, -0.28209937, ..., -0.32826333,
         0.67709663, -0.16314854],
       [-1.70452544, -0.34704629, -0.28209937, ..., -0.32826333,
         0.67709663, -0.16314854],
       [ 0.08301176, -0.34704629, -0.14334471, ..., -0.32826333,
         0.67709663, -0.16314854],
       ...,
       [ 0.49552034, -0.34704629,  2.51415978, ..., -0.32826333,
         0.67709663, -0.16314854],
       [ 0.35801748, -0.34704629, -0.2797476 , ..., -0.32826333,
         0.67709663,  6.12938358],
       [-1.01701113, -0.34393854,  0.23587883, ..., -0.32826333,
         0.67709663, -0.16314854]], shape=(5284, 19))

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [15]:
datasets = [
    {"name": "raw",        "X_train": X_train,      "X_test": X_test},
    {"name": "scaled",        "X_train": X_train_scaled,      "X_test": X_test_scaled}
]

In [16]:
models = [
    {"name": "LR",               "model": LogisticRegression(max_iter=1000)},
    {"name": "Bagging",          "model": BaggingClassifier(estimator=DecisionTreeClassifier())},
    {"name": "RandomForest",     "model": RandomForestClassifier()},
    {"name": "AdaBoost",         "model": AdaBoostClassifier(estimator=DecisionTreeClassifier())},
    {"name": "GradientBoosting", "model": GradientBoostingClassifier()},
    {"name": "XGBoost",          "model": XGBClassifier()}
]

- Evaluate your model

In [17]:
results = []

for ds in datasets:
    for m in models:
        m["model"].fit(ds["X_train"], y_train)
        pred = m["model"].predict(ds["X_test"])

        results.append({
            "dataset": ds["name"],
            "model":   m["name"],
            "F1Score":    f1_score(y_test, pred),
        })
        print(f"{ds['name']:>10} | {m['name']:<16} done")

results_df = pd.DataFrame(results).sort_values("F1Score", ascending=False).reset_index(drop=True)
results_df

c:\Users\glint\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


       raw | LR               done
       raw | Bagging          done
       raw | RandomForest     done
       raw | AdaBoost         done
       raw | GradientBoosting done
       raw | XGBoost          done
    scaled | LR               done
    scaled | Bagging          done
    scaled | RandomForest     done
    scaled | AdaBoost         done
    scaled | GradientBoosting done
    scaled | XGBoost          done


,dataset,model,F1Score
0,raw,GradientBoosting,0.795948
1,scaled,GradientBoosting,0.795948
2,raw,XGBoost,0.793103
3,scaled,XGBoost,0.793103
4,scaled,LR,0.789238
5,raw,LR,0.787155
6,scaled,RandomForest,0.782542
7,raw,Bagging,0.773541
8,raw,RandomForest,0.771957
9,scaled,Bagging,0.770656


**Grid/Random Search**

- Run Random Search

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.base import clone
from scipy.stats import randint, uniform

param_distributions = {
    "LR": {
        "C": uniform(0.01, 100),
        "solver": ["lbfgs", "liblinear"],
        "max_iter": [1000]
    },

    "Bagging": {
        "n_estimators": randint(10, 200),
        "max_samples": uniform(0.5, 0.5),
        "max_features": uniform(0.5, 0.5)
    },

    "RandomForest": {
        "n_estimators": randint(100, 500),
        "max_depth": [None, 5, 10, 20, 30],
        "min_samples_split": randint(2, 20),
        "min_samples_leaf": randint(1, 10),
        "max_features": ["sqrt", "log2"]
    },

    "AdaBoost": {
        "n_estimators": randint(50, 300),
        "learning_rate": uniform(0.01, 1.0)
    },

    "GradientBoosting": {
        "n_estimators": randint(50, 300),
        "learning_rate": uniform(0.01, 0.3),
        "max_depth": randint(2, 10),
        "subsample": uniform(0.7, 0.3)
    },

    "XGBoost": {
        "n_estimators": randint(100, 500),
        "max_depth": randint(3, 10),
        "learning_rate": uniform(0.01, 0.3),
        "subsample": uniform(0.7, 0.3),
        "colsample_bytree": uniform(0.7, 0.3)
    }
}

In [25]:
# Random Search
best_models = {}

for m in models:

    name = m["name"]
    model = clone(m["model"])

    print(f"\nRunning Random Search for {name}...")

    rs = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions[name],
        n_iter=20,
        cv=5,
        scoring="f1",
        n_jobs=-1,
        random_state=42
    )

    rs.fit(X_train, y_train)

    best_models[name] = rs.best_estimator_

    print("Best score:", rs.best_score_)
    print("Best params:", rs.best_params_)


Running Random Search for LR...
Best score: 0.8008575512575096
Best params: {'C': np.float64(0.08787658410143283), 'max_iter': 1000, 'solver': 'liblinear'}

Running Random Search for Bagging...
Best score: 0.794312988080021
Best params: {'max_features': np.float64(0.6912309956335814), 'max_samples': np.float64(0.9916154429033941), 'n_estimators': 140}

Running Random Search for RandomForest...
Best score: 0.8115233640420263
Best params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 8, 'min_samples_split': 17, 'n_estimators': 336}

Running Random Search for AdaBoost...
Best score: 0.7610306368027587
Best params: {'learning_rate': np.float64(0.20967378215835974), 'n_estimators': 104}

Running Random Search for GradientBoosting...
Best score: 0.8111667656058271
Best params: {'learning_rate': np.float64(0.06502135295603015), 'max_depth': 5, 'n_estimators': 107, 'subsample': np.float64(0.8574269294896713)}

Running Random Search for XGBoost...
Best score: 0.8138526284267854

In [34]:

results = []

for name, model in best_models.items():

    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

results_df1 = pd.DataFrame(results).sort_values("F1", ascending=False)

results_df1

,Model,Accuracy,Precision,Recall,F1
4,GradientBoosting,0.788956,0.772857,0.818457,0.795004
5,XGBoost,0.787443,0.774566,0.810893,0.792313
0,LR,0.788200,0.778917,0.804841,0.791667
1,Bagging,0.789713,0.787106,0.794251,0.790663
2,RandomForest,0.783661,0.777778,0.794251,0.785928
3,AdaBoost,0.756430,0.734440,0.803328,0.767341


- Run Grid Search

In [32]:
def get_grid(model_name):
    """
    Returns hyperparameter grid depending on the model.
    """

    if model_name == "RandomForest":
        return {
            "n_estimators": [100, 200, 300],
            "max_depth": [None, 10, 20, 30],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }

    elif model_name == "GradientBoosting":
        return {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [3, 5, 7],
            "subsample": [0.7, 0.8, 1.0]
        }

    elif model_name == "LR":
        return {
            "C": [0.01, 0.1, 1, 10, 100],
            "penalty": ["l2"],
            "solver": ["lbfgs"],
            "max_iter": [1000]
        }

    elif model_name == "XGBoost":
        return {
            "n_estimators": [100, 200, 300],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.01, 0.05, 0.1],
            "subsample": [0.7, 0.8, 1.0],
            "colsample_bytree": [0.7, 0.8, 1.0]
        }

    elif model_name == "AdaBoost":
        return {
            "n_estimators": [50, 100, 200],
            "learning_rate": [0.01, 0.1, 1.0]
        }

    elif model_name == "Bagging":
        return {
            "n_estimators": [10, 50, 100],
            "max_samples": [0.5, 0.7, 1.0],
            "max_features": [0.5, 0.7, 1.0]
        }

    return {}

In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone

grid_results = {}

for m in models:

    name = m["name"]
    model = clone(m["model"])

    print(f"\nRunning Grid Search for {name}...")

    grid = get_grid(name)

    if grid:

        gs = GridSearchCV(
            estimator=model,
            param_grid=grid,
            cv=5,
            scoring="f1",
            n_jobs=-1
        )

        gs.fit(X_train, y_train)

        grid_results[name] = gs.best_estimator_

        print(name)
        print("Best score:", gs.best_score_)
        print("Best params:", gs.best_params_)


Running Grid Search for LR...


c:\Users\glint\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


LR
Best score: 0.8130755676870534
Best params: {'C': 0.01, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'lbfgs'}

Running Grid Search for Bagging...
Bagging
Best score: 0.7932487438105771
Best params: {'max_features': 0.7, 'max_samples': 1.0, 'n_estimators': 50}

Running Grid Search for RandomForest...
RandomForest
Best score: 0.8126736238226284
Best params: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}

Running Grid Search for AdaBoost...
AdaBoost
Best score: 0.7583985477852033
Best params: {'learning_rate': 1.0, 'n_estimators': 200}

Running Grid Search for GradientBoosting...
GradientBoosting
Best score: 0.8160329743254662
Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}

Running Grid Search for XGBoost...
XGBoost
Best score: 0.8183946362239588
Best params: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.7}


- Evaluate your model

In [30]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results = []

for name, model in grid_results.items():

    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)

results_df

,Model,Accuracy,Precision,Recall,F1
3,GradientBoosting,0.786687,0.760660,0.836611,0.796830
0,Bagging,0.794251,0.799692,0.785174,0.792366
4,XGBoost,0.784418,0.765537,0.819970,0.791819
1,RandomForest,0.785174,0.776799,0.800303,0.788376
2,AdaBoost,0.754917,0.732414,0.803328,0.766234
